In [13]:
import os
os.environ["GLOG_minloglevel"] = "3"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cv2
import mediapipe as mp
from mediapipe.python.solutions import drawing_utils, drawing_styles, hands
from dollarpy import Point, Recognizer, Template
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import deque, defaultdict
import re
import time

mp_drawing = drawing_utils
mp_drawing_styles = drawing_styles
mp_hands = hands
print(f"MediaPipe loaded from: {mp.__file__}")

MediaPipe loaded from: /Users/abdelrahmannabil/Downloads/Smart_Shopping-MK_Branch/.venv/lib/python3.11/site-packages/mediapipe/__init__.py


In [14]:
# Global configuration for both-hand gesture tracking and classification
CAMERA_INDEX = 0
MIN_DETECTION_CONFIDENCE = 0.5
MIN_TRACKING_CONFIDENCE = 0.5
CLASSIFICATION_THRESHOLD = 0.05
WINDOW_SIZE = 45

# Trajectory feature for dollarpy ($1-style input)
TRACK_LANDMARK_ID = 8  # index fingertip
LEFT_HAND_STROKE_ID = 1
RIGHT_HAND_STROKE_ID = 101
MIN_POINT_SPREAD = 0.002

templates: List[Template] = []

In [15]:
def extract_hand_points(hand_results) -> List[Point]:
    """Extract hand trajectories and assign side using MediaPipe handedness."""
    frame_points: List[Point] = []

    if not hand_results.multi_hand_landmarks:
        return frame_points

    for hand_idx, hand_landmarks in enumerate(hand_results.multi_hand_landmarks):
        tip = hand_landmarks.landmark[TRACK_LANDMARK_ID]

        is_left_hand = False
        if hand_results.multi_handedness and hand_idx < len(hand_results.multi_handedness):
            label = hand_results.multi_handedness[hand_idx].classification[0].label.lower()
            is_left_hand = label == "left"

        stroke_id = LEFT_HAND_STROKE_ID if is_left_hand else RIGHT_HAND_STROKE_ID
        frame_points.append(Point(tip.x, tip.y, stroke_id))

    return frame_points


def normalize_label(name: str) -> str:
    """Normalize filename into a stable class label token."""
    label = re.sub(r"[^a-zA-Z0-9]+", "_", name.strip().lower())
    return re.sub(r"_+", "_", label).strip("_")


def get_points(
    source,
    label: str,
    show_window: bool = False,
    draw_landmarks: bool = True,
    max_frames: Optional[int] = None,
    camera_index: int = CAMERA_INDEX,
) -> List[Point]:
    """Extract hand trajectory points from video file path or webcam source."""
    if source == "camera":
        cap = cv2.VideoCapture(camera_index)
    else:
        cap = cv2.VideoCapture(str(source))

    if not cap.isOpened():
        raise ValueError(f"Could not open source: {source}")

    all_points: List[Point] = []
    frame_count = 0

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=MIN_DETECTION_CONFIDENCE,
        min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
    ) as hand_tracker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            hand_results = hand_tracker.process(image_rgb)
            image_rgb.flags.writeable = True
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            frame_points = extract_hand_points(hand_results)
            all_points.extend(frame_points)

            if draw_landmarks and hand_results.multi_hand_landmarks:
                for hand_landmarks in hand_results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        image_bgr,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                    )

            if show_window:
                cv2.imshow(label, image_bgr)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

            frame_count += 1
            if max_frames is not None and frame_count >= max_frames:
                break

    cap.release()
    if show_window:
        cv2.destroyAllWindows()
        cv2.waitKey(50)

    if not all_points:
        raise ValueError(f"No trackable hand points found for: {label}")

    x_values = [p.x for p in all_points]
    y_values = [p.y for p in all_points]
    spread = max(max(x_values) - min(x_values), max(y_values) - min(y_values))
    if spread < MIN_POINT_SPREAD:
        raise ValueError(f"Insufficient motion spread for: {label}")

    return all_points


def build_templates(dataset_map: Dict[str, List[str]], preview: bool = False) -> List[Template]:
    """Build template list from class label -> list of video paths."""
    built_templates: List[Template] = []
    for class_label, video_paths in dataset_map.items():
        for idx, video_path in enumerate(video_paths, start=1):
            if not Path(video_path).exists():
                print(f"Skipped (missing file): {class_label} sample {idx} -> {video_path}")
                continue

            sample_label = f"{class_label}_s{idx}"
            try:
                points = get_points(video_path, sample_label, show_window=preview)
            except ValueError as exc:
                print(f"Skipped (bad sample): {sample_label} -> {exc}")
                continue

            built_templates.append(Template(class_label, points))
            print(f"Loaded template: {class_label} sample {idx}")

    if not built_templates:
        raise ValueError("No templates built. Verify training clips and motion spread.")

    return built_templates


def classify_points(
    recognizer: Recognizer,
    points: List[Point],
    threshold: float = CLASSIFICATION_THRESHOLD,
    unknown_label: str = "unknown",
) -> Tuple[str, float]:
    """Classify points with thresholding for uncertain predictions."""
    try:
        prediction = recognizer.recognize(points)
    except ZeroDivisionError:
        return unknown_label, 0.0

    predicted_label = prediction[0]
    score = prediction[1] if len(prediction) > 1 else 0.0
    if score < threshold:
        return unknown_label, score
    return predicted_label, score


def classify_video(recognizer: Recognizer, video_path: str, label: str = "video_test") -> Tuple[str, float, float]:
    start = time.time()
    points = get_points(video_path, label, show_window=False)
    predicted_label, score = classify_points(recognizer, points)
    latency = time.time() - start
    return predicted_label, score, latency


def run_realtime_classification(
    recognizer: Recognizer,
    window_size: int = WINDOW_SIZE,
    camera_index: int = CAMERA_INDEX,
    threshold: float = CLASSIFICATION_THRESHOLD,
) -> None:
    """Run webcam tracking and classify hand trajectory windows."""
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise ValueError(f"Could not open camera index: {camera_index}")

    point_buffer = deque(maxlen=window_size)

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=MIN_DETECTION_CONFIDENCE,
        min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
    ) as hand_tracker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            hand_results = hand_tracker.process(image_rgb)
            image_rgb.flags.writeable = True
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            frame_points = extract_hand_points(hand_results)
            point_buffer.extend(frame_points)

            prediction_text = "Collecting frames..."
            if len(point_buffer) >= point_buffer.maxlen:
                pred_label, pred_score = classify_points(
                    recognizer,
                    list(point_buffer),
                    threshold=threshold,
                )
                prediction_text = f"{pred_label} ({pred_score:.3f})"

            if hand_results.multi_hand_landmarks:
                for hand_landmarks in hand_results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        image_bgr,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                    )

            cv2.putText(
                image_bgr,
                prediction_text,
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2,
                cv2.LINE_AA,
            )
            cv2.imshow("Realtime Classification", image_bgr)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(50)


def getPoints(videoURL, label):
    """Backward-compatible wrapper for existing cells."""
    return get_points(videoURL, label, show_window=True)

In [16]:
# Classification safety checks are integrated in Cell 3.
print("Using Cell 3 for extraction/classification pipeline.")

Using Cell 3 for extraction/classification pipeline.


In [17]:
# Quick extraction sanity check using right-hand training data
sample_video = "Right Hand/Clockwise/Train/Train 1.mov"
sample_points = get_points(sample_video, "sanity_check", show_window=False)
print(f"Extracted points: {len(sample_points)}")

I0000 00:00:1774204234.438804 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204234.468433 1418997 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204234.479150 1418997 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Extracted points: 133


In [18]:
# Build templates from both hands using Train/Test split in each movement folder
LEFT_HAND_BASE_FOLDER = "Left Hand"
RIGHT_HAND_BASE_FOLDER = "Right Hand"
VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv")

# Folder-to-label corrections (swipe folders were mapped in reverse).
MOVEMENT_LABEL_SWAP = {
    "swipe_left": "swipe_right",
    "swipe_right": "swipe_left",
}

TRAIN_MAP: Dict[str, List[str]] = {}
TEST_MAP: Dict[str, str] = {}

HAND_DATASETS = [
    (LEFT_HAND_BASE_FOLDER, "left_hand"),
    (RIGHT_HAND_BASE_FOLDER, "right_hand"),
]

for base_folder, hand_suffix in HAND_DATASETS:
    base_path = Path(base_folder)
    if not base_path.exists():
        print(f"Warning: folder not found -> {base_folder}")
        continue

    for movement_dir in sorted(base_path.iterdir()):
        if not movement_dir.is_dir():
            continue

        movement_label = normalize_label(movement_dir.name)
        movement_label = MOVEMENT_LABEL_SWAP.get(movement_label, movement_label)
        class_label = f"{movement_label}_{hand_suffix}"

        train_dir = movement_dir / "Train"
        test_dir = movement_dir / "Test"

        train_files = []
        test_files = []

        if train_dir.exists():
            train_files = sorted(
                [p for p in train_dir.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
            )

        if test_dir.exists():
            test_files = sorted(
                [p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
            )

        if len(train_files) == 0:
            print(f"Skipped class with no train files: {class_label}")
            continue

        TRAIN_MAP[class_label] = [str(p) for p in train_files]

        if len(test_files) > 0:
            TEST_MAP[class_label] = str(test_files[0])

        print(f"{class_label}: train={len(train_files)} test={len(test_files)}")
        if len(train_files) != 2 or len(test_files) != 1:
            print("  Warning: expected 2 train videos and 1 test video for this class.")

if not TRAIN_MAP:
    raise ValueError("No training videos found. Check Left Hand and Right Hand folder structures.")

DATASET_MAP = TRAIN_MAP
templates = build_templates(DATASET_MAP, preview=False)
recognizer = Recognizer(templates)
print(f"Templates ready: {len(templates)}")
print(f"Test classes ready: {len(TEST_MAP)}")

anticlockwise_left_hand: train=2 test=1
clockwise_left_hand: train=2 test=1
select_tab_left_hand: train=2 test=1
swipe_left_left_hand: train=2 test=1
swipe_right_left_hand: train=2 test=1
anticlockwise_right_hand: train=2 test=1
clockwise_right_hand: train=2 test=1
select_tab_right_hand: train=2 test=1
swipe_left_right_hand: train=2 test=1
swipe_right_right_hand: train=2 test=1


I0000 00:00:1774204238.488206 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204238.497934 1419104 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204238.505770 1419104 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Loaded template: anticlockwise_left_hand sample 1


I0000 00:00:1774204241.322338 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204241.329965 1419135 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204241.335619 1419135 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Loaded template: anticlockwise_left_hand sample 2


...

Loaded template: swipe_left_right_hand sample 2


I0000 00:00:1774204292.373229 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204292.381000 1419845 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204292.387418 1419845 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Loaded template: swipe_right_right_hand sample 1


I0000 00:00:1774204295.414227 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204295.421222 1419902 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204295.429197 1419902 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Loaded template: swipe_right_right_hand sample 2
Templates ready: 20
Test classes ready: 10


In [19]:
# Export trained DollarPy templates to a .pth file
# Note: this is a serialized template-based recognizer state, not a PyTorch neural network.
from pathlib import Path
import pickle

MODEL_EXPORT_PATH = Path("models/gesture_recognizer_dollarpy.pth")
MODEL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "format": "dollarpy_templates_v1",
    "class_names": sorted(TRAIN_MAP.keys()),
    "train_map": TRAIN_MAP,
    "test_map": TEST_MAP,
    "templates": templates,
    "config": {
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "window_size": WINDOW_SIZE,
        "min_detection_confidence": MIN_DETECTION_CONFIDENCE,
        "min_tracking_confidence": MIN_TRACKING_CONFIDENCE,
        "track_landmark_id": TRACK_LANDMARK_ID,
    },
}

# Try torch.save for a standard .pth workflow; fallback to pickle if torch is unavailable.
saved_with = "pickle"
try:
    import torch
    torch.save(payload, MODEL_EXPORT_PATH)
    saved_with = "torch.save"
except Exception:
    with MODEL_EXPORT_PATH.open("wb") as f:
        pickle.dump(payload, f)

print(f"Saved model payload to: {MODEL_EXPORT_PATH.resolve()}")
print(f"Serializer used: {saved_with}")
print(f"Classes: {len(payload['class_names'])}")

Saved model payload to: /Users/abdelrahmannabil/Downloads/Smart_Shopping-MK_Branch/models/gesture_recognizer_dollarpy.pth
Serializer used: torch.save
Classes: 10


In [20]:
# Offline evaluation on both-hand test videos (1 test sample per movement/class)
if not TEST_MAP:
    raise ValueError("TEST_MAP is empty. Run the training cell first and verify Test folders.")

results = []
for class_label, test_video in sorted(TEST_MAP.items()):
    try:
        predicted_label, score, latency = classify_video(
            recognizer,
            test_video,
            label=f"{class_label}_test",
        )
        is_correct = predicted_label == class_label
        results.append((class_label, predicted_label, score, latency, is_correct))
    except ValueError as exc:
        print(f"Skipped test sample for {class_label}: {exc}")

if not results:
    raise ValueError("No valid test samples were evaluated.")

print("Per-class test results:")
for expected, predicted, score, latency, is_correct in results:
    print(
        f"expected={expected} | predicted={predicted} | "
        f"score={score:.4f} | latency={latency:.3f}s | correct={is_correct}"
    )

correct_count = sum(1 for _, _, _, _, ok in results if ok)
accuracy = correct_count / len(results) if results else 0.0
print(f"Test accuracy: {correct_count}/{len(results)} = {accuracy:.2%}")

I0000 00:00:1774204298.172957 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204298.182163 1419935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204298.189006 1419935 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1774204302.229349 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204302.235673 1419981 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204302.240896 1419981 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1774204305.725629 1395706 gl_context

Per-class test results:
expected=anticlockwise_left_hand | predicted=anticlockwise_left_hand | score=0.2243 | latency=4.068s | correct=True
expected=anticlockwise_right_hand | predicted=anticlockwise_right_hand | score=0.5565 | latency=3.495s | correct=True
expected=clockwise_left_hand | predicted=clockwise_left_hand | score=0.3583 | latency=3.079s | correct=True
expected=clockwise_right_hand | predicted=anticlockwise_right_hand | score=0.6182 | latency=3.814s | correct=False
expected=select_tab_left_hand | predicted=select_tab_right_hand | score=0.4232 | latency=2.536s | correct=False
expected=select_tab_right_hand | predicted=select_tab_right_hand | score=0.5618 | latency=2.982s | correct=True
expected=swipe_left_left_hand | predicted=swipe_left_right_hand | score=0.4167 | latency=2.363s | correct=False
expected=swipe_left_right_hand | predicted=swipe_left_right_hand | score=0.3118 | latency=2.309s | correct=True
expected=swipe_right_left_hand | predicted=swipe_right_right_hand | sco

In [21]:
# Real-time demo (press q to quit the window)
run_realtime_classification(recognizer, window_size=WINDOW_SIZE, camera_index=CAMERA_INDEX)
print("Realtime demo function is ready. Uncomment run_realtime_classification(...) to start.")

I0000 00:00:1774204332.615784 1395706 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1774204332.626995 1420510 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774204332.633612 1420510 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Realtime demo function is ready. Uncomment run_realtime_classification(...) to start.


## Submission Checklist: MediaPipe + DollarPy (Both Hands)

 > Criterion: Track hand skeleton using MediaPipe and classify movement using DollarPy.

1. Keep both datasets active: Left Hand and Right Hand.
2. For each movement folder in each hand, keep split as Train and Test.
3. Put exactly 2 videos in Train and 1 video in Test for each movement.
4. Run setup and helper cells.
5. Run the training cell to build TRAIN_MAP and TEST_MAP automatically for both hands.
6. Run offline evaluation to test one sample per movement from Test folders.
7. Run real-time classification using webcam.

### Notes
- Class labels are generated as movement_left_hand and movement_right_hand.
- Hand side is inferred using MediaPipe handedness labels.
- Swipe left/right label swap correction is applied in the training/evaluation cell.
- Cell output warns if any movement does not have 2 train and 1 test videos.
- Press q to close any OpenCV window.